<a href="https://colab.research.google.com/github/Fqrina/ai-archive/blob/main/news_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

print("Libraries imported successfully!")

Libraries imported successfully!


# STEP 1: LOAD DATA

In [ ]:
print("1. LOAD DATA")

train_df = pd.read_csv('train (12).csv')
test_df = pd.read_csv('test (12).csv')

print(f"Train data shape: {train_df.shape}")
print("\nTrain data head():")
print(train_df.head())

print(f"\nTest data shape: {test_df.shape}")
print("\nTest data head():")
print(test_df.head())

print("\nTarget distribution:")
print(train_df['label'].value_counts().sort_index())

1. LOAD DATA
Train data shape: (100, 3)

Train data head():
   id                                               text  label
0   0  The agency confirmed that the lunar probe tran...      0
1   1  Developers said that the team benchmarked the ...      2
2   2  The minister stated that the government debate...      3
3   3  Officials said that the debate centered on tax...      3
4   4  Analysts reported that the election campaign f...      3

Test data shape: (20, 2)

Test data head():
   id                                               text
0   0  A new briefing said that astronauts prepared f...
1   1  Officials said that the senate vote moved the ...
2   2  A committee update showed that the election ca...
3   3  Scientists reported that the lunar probe trans...
4   4  The coach explained that the stadium crowd rea...

Target distribution:
label
0    22
1    27
2    26
3    25
Name: count, dtype: int64


# STEP 2: EXPLORATORY DATA ANALYSIS (BASIC)

In [ ]:
print("2. BASIC EDA")

print("\nData types:")
print(train_df.dtypes)

print("\nMissing values:")
print(train_df.isnull().sum())

print("\nText sample:")
print(train_df['text'].iloc[0])

2. BASIC EDA

Data types:
id        int64
text     object
label     int64
dtype: object

Missing values:
id       0
text     0
label    0
dtype: int64

Text sample:
The agency confirmed that the lunar probe transmitted fresh telemetry data. The report highlighted satellite developments and kept the tone short and factual.


# STEP 3: PREPARE DATA

In [ ]:
def normal(x):
  map = {".": "", ",":"", "-":""}
  x = x.lower()
  for long,new in map.items():
    x = x.replace(long, new)
  return x

In [ ]:
train_df.head()

,id,text,label
0,0,The agency confirmed that the lunar probe tran...,0
1,1,Developers said that the team benchmarked the ...,2
2,2,The minister stated that the government debate...,3
3,3,Officials said that the debate centered on tax...,3
4,4,Analysts reported that the election campaign f...,3


In [ ]:
train_df["text"] = train_df["text"].apply(normal)
train_df.head()

,id,text,label
0,0,the agency confirmed that the lunar probe tran...,0
1,1,developers said that the team benchmarked the ...,2
2,2,the minister stated that the government debate...,3
3,3,officials said that the debate centered on tax...,3
4,4,analysts reported that the election campaign f...,3


In [ ]:
print("3. PREPARE DATA")

X = train_df[['text']]
y = train_df['label']

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")

3. PREPARE DATA
Training set: (80, 1)
Validation set: (20, 1)


In [ ]:
vect = TfidfVectorizer()
x_out = vect.fit_transform(X_train["text"])
val_out = vect.transform(X_val["text"])
test_out = vect.transform(test_df["text"])

In [ ]:
x_out.shape

(80, 146)

# STEP 4: EXAMPLE MODEL

In [ ]:
print("4. BASELINE MODEL")

dummy_model = DummyClassifier(strategy='most_frequent')
dummy_model.fit(x_out, y_train)
log = LogisticRegression()
log.fit(x_out, y_train)
rfc = RandomForestClassifier()
rfc.fit(x_out, y_train)
hgb = HistGradientBoostingClassifier()
hgb.fit(x_out.toarray(), y_train)


4. BASELINE MODEL


HistGradientBoostingClassifier()

In [ ]:
v = SVC()
v.fit(x_out, y_train)

SVC()

In [ ]:

val_predictions = v.predict(val_out)
val_accuracy = accuracy_score(y_val, val_predictions)
val_f1 = f1_score(y_val, val_predictions, average='macro')

print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Validation Macro F1: {val_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_val, val_predictions))

Validation Accuracy: 1.0000
Validation Macro F1: 1.0000

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         5
           1       1.00      1.00      1.00         5
           2       1.00      1.00      1.00         5
           3       1.00      1.00      1.00         5

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20



# STEP 5: GENERATE SUBMISSION

In [ ]:
print("5. FINAL SUBMISSION")

test_predictions = v.predict(test_out)

submission = pd.DataFrame({
    'id': test_df['id'],
    'label': test_predictions,
})

print("Submission preview:")
print(submission.head(10))
print(f"\nSubmission shape: {submission.shape}")
print("\nPrediction distribution:")
print(pd.Series(test_predictions).value_counts().sort_index())

submission.to_csv('submission3.csv', index=False)
print("\n✓ submission.csv berhasil dibuat!")

5. FINAL SUBMISSION
Submission preview:
   id  label
0   0      0
1   1      3
2   2      3
3   3      0
4   4      1
5   5      2
6   6      0
7   7      2
8   8      3
9   9      2

Submission shape: (20, 2)

Prediction distribution:
0    8
1    3
2    4
3    5
Name: count, dtype: int64

✓ submission.csv berhasil dibuat!
